# 🕵️ Dia 13 — Script Bash para automatizar uma etapa de reconhecimento

**Investigador Cyber 90 Dias** · Fase 1 · Rank Jutsu `D`

> Baseado no *Day 13* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-1-foundations/day-13.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Terça, 9h20. `coopserra-azul.example` agora aponta para `203.0.113.78`, e `serraazul-seguranca.example` ganhou um `MX`. Desde quando? Seu caderno de segunda à noite não diz: você esqueceu o `MX` e usou dois resolvedores diferentes.

Rafa: *"Escreve um script. Toda manhã, igual, com hash. Aí a diferença é do mundo."*

Neste notebook você vai:

1. **Reescrever o `dns-snapshot.sh` em Python stdlib**, com o mesmo formato de saída.
2. **Conferir os hashes** dos dois retratos da Serra Azul e rodar o diff.
3. **Jogar o quiz do `set -euo pipefail`**.
4. **Gerar o relatório de mudanças** do caso.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo funciona **sem internet**; quando uma célula tenta a rede, ela tem plano B embutido.

## 1. O `dns-snapshot.sh`, em Python

O Colab não tem `dig`. Duas saídas sem biblioteca externa:

- `socket.getaddrinfo` (o resolvedor do sistema): só responde A e AAAA;
- **DNS sobre HTTPS em JSON** (`dns.google/resolve`): responde qualquer tipo, por `urllib`.

As duas ficam em `try/except`. Sem rede, o notebook usa uma resposta de exemplo embutida, e diz isso na tela.

In [ ]:
import json, socket, urllib.request, hashlib, time
from datetime import datetime, timezone
from pathlib import Path

RESOLVEDOR = "dns.google (DoH JSON)"            # registre SEMPRE qual resolvedor foi usado
TIPOS = ["A", "AAAA", "NS", "MX", "TXT", "CAA"]  # os mesmos 6 tipos do script do original

# Plano B: o que o DoH devolveu para estes nomes num teste (formato reduzido: tipo -> [(ttl, valor)])
EXEMPLO_OFFLINE = {
    "example.com": {"MX": [(241, "0 .")], "TXT": [(300, '"v=spf1 -all"')]},
}

def doh(nome, tipo):
    """Consulta DoH JSON. Devolve [(ttl, valor)] ou levanta exceção se não houver rede."""
    url = f"https://dns.google/resolve?name={nome}&type={tipo}"
    with urllib.request.urlopen(url, timeout=6) as r:
        dados = json.load(r)
    numero = {"A": 1, "NS": 2, "MX": 15, "TXT": 16, "AAAA": 28, "CAA": 257}[tipo]
    # Igual ao filtro $4 == rt do awk: descarta CNAME e outras linhas que não são do tipo pedido
    return [(a["TTL"], a["data"]) for a in dados.get("Answer", []) if a.get("type") == numero]

def snapshot(dominios, pasta_raiz="snapshots"):
    """Mesmo contrato do script: pasta nova com hora UTC, dns.csv, dns.stable.txt e SHA256SUMS."""
    run_ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    out = Path(pasta_raiz) / run_ts
    out.mkdir(parents=True, exist_ok=False)          # exist_ok=False: nunca sobrescreve uma coleta
    csv_linhas = ["collected_utc,domain,rtype,ttl,value,resolver"]
    estavel = set()
    offline = False
    for d in dominios:
        for t in TIPOS:
            agora = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
            try:
                respostas = doh(d, t)
            except Exception:
                offline = True
                respostas = EXEMPLO_OFFLINE.get(d, {}).get(t, [])   # plano B
            for ttl, valor in respostas:
                estavel.add(f"{d}\t{t}\t{valor}")                 # cópia sem TTL para o diff
                v = valor.replace('"', '""')                         # regra do CSV: aspas dobradas
                csv_linhas.append(f'{agora},{d},{t},{ttl},"{v}",{RESOLVEDOR}')
        time.sleep(0.2)                                              # educado com o resolvedor
    (out / "dns.csv").write_text("\n".join(csv_linhas) + "\n", encoding="utf-8")
    (out / "dns.stable.txt").write_text("\n".join(sorted(estavel)) + "\n", encoding="utf-8")
    somas = [f"{hashlib.sha256((out / f).read_bytes()).hexdigest()}  {f}" for f in ("dns.csv", "dns.stable.txt")]
    (out / "SHA256SUMS").write_text("\n".join(somas) + "\n", encoding="utf-8")
    print(f"wrote {len(csv_linhas) - 1} records to {out / 'dns.csv'}" + ("  (OFFLINE: resposta de exemplo)" if offline else ""))
    return out

# Só domínios públicos de prática, como no Passo 1 do original
pasta = snapshot(["example.com"])
print((pasta / "dns.csv").read_text())

> 💡 Repare no `exist_ok=False`: se você rodar duas vezes no mesmo segundo, a segunda rodada **falha** em vez de apagar a primeira. É o equivalente Python do "nunca sobrescreva uma coleta anterior".

## 2. ➕ Os dois retratos da Serra Azul

Os domínios `.example` não resolvem na internet; os retratos do caso foram simulados no formato exato do script. A célula baixa os seis arquivos do repositório (ou usa a cópia embutida) e os grava em `snapshots-isca/`.

In [ ]:
EMBUTIDO = {'20261005T210005Z': {'dns.csv': 'collected_utc,domain,rtype,ttl,value,resolver\n2026-10-05T21:00:05Z,serraazul.example,A,300,"198.51.100.20",1.1.1.1\n2026-10-05T21:00:05Z,serraazul.example,MX,3600,"10 mx1.provedor-email.example.",1.1.1.1\n2026-10-05T21:00:05Z,serraazul.example,NS,86400,"ns1.provedor-dns.example.",1.1.1.1\n2026-10-05T21:00:05Z,serraazul.example,NS,86400,"ns2.provedor-dns.example.",1.1.1.1\n2026-10-05T21:00:05Z,serraazul.example,TXT,3600,"""v=spf1 include:_spf.provedor-email.example include:envios.boletim.example ~all""",1.1.1.1\n2026-10-05T21:00:05Z,serraazul-pix.example,A,300,"203.0.113.77",1.1.1.1\n2026-10-05T21:00:05Z,serraazul-pix.example,NS,3600,"ns1.dnsbarato.example.",1.1.1.1\n2026-10-05T21:00:05Z,serraazul-pix.example,NS,3600,"ns2.dnsbarato.example.",1.1.1.1\n2026-10-05T21:00:05Z,serraazul-seguranca.example,A,300,"203.0.113.78",1.1.1.1\n2026-10-05T21:00:05Z,serraazul-seguranca.example,NS,3600,"ns1.dnsbarato.example.",1.1.1.1\n2026-10-05T21:00:05Z,serraazul-seguranca.example,NS,3600,"ns2.dnsbarato.example.",1.1.1.1\n2026-10-05T21:00:05Z,coopserra-azul.example,NS,3600,"ns1.dnsbarato.example.",1.1.1.1\n2026-10-05T21:00:05Z,coopserra-azul.example,NS,3600,"ns2.dnsbarato.example.",1.1.1.1\n', 'dns.stable.txt': 'coopserra-azul.example\tNS\tns1.dnsbarato.example.\ncoopserra-azul.example\tNS\tns2.dnsbarato.example.\nserraazul-pix.example\tA\t203.0.113.77\nserraazul-pix.example\tNS\tns1.dnsbarato.example.\nserraazul-pix.example\tNS\tns2.dnsbarato.example.\nserraazul-seguranca.example\tA\t203.0.113.78\nserraazul-seguranca.example\tNS\tns1.dnsbarato.example.\nserraazul-seguranca.example\tNS\tns2.dnsbarato.example.\nserraazul.example\tA\t198.51.100.20\nserraazul.example\tMX\t10 mx1.provedor-email.example.\nserraazul.example\tNS\tns1.provedor-dns.example.\nserraazul.example\tNS\tns2.provedor-dns.example.\nserraazul.example\tTXT\t"v=spf1 include:_spf.provedor-email.example include:envios.boletim.example ~all"\n', 'SHA256SUMS': 'f4b0f9602aabbbc649beaa759080673f54f068cef74aa055ebab3dec371e197b  dns.csv\n5326acdfaccb2b6fe4876d79f2bb3bddff8cbc31d65498a5a33db7dc3fd13ea3  dns.stable.txt\n'}, '20261006T090004Z': {'dns.csv': 'collected_utc,domain,rtype,ttl,value,resolver\n2026-10-06T09:00:04Z,serraazul.example,A,300,"198.51.100.20",1.1.1.1\n2026-10-06T09:00:04Z,serraazul.example,MX,3600,"10 mx1.provedor-email.example.",1.1.1.1\n2026-10-06T09:00:04Z,serraazul.example,NS,86400,"ns1.provedor-dns.example.",1.1.1.1\n2026-10-06T09:00:04Z,serraazul.example,NS,86400,"ns2.provedor-dns.example.",1.1.1.1\n2026-10-06T09:00:04Z,serraazul.example,TXT,3600,"""v=spf1 include:_spf.provedor-email.example include:envios.boletim.example ~all""",1.1.1.1\n2026-10-06T09:00:04Z,serraazul-pix.example,A,300,"203.0.113.77",1.1.1.1\n2026-10-06T09:00:04Z,serraazul-pix.example,NS,3600,"ns1.dnsbarato.example.",1.1.1.1\n2026-10-06T09:00:04Z,serraazul-pix.example,NS,3600,"ns2.dnsbarato.example.",1.1.1.1\n2026-10-06T09:00:04Z,serraazul-seguranca.example,A,300,"203.0.113.78",1.1.1.1\n2026-10-06T09:00:04Z,serraazul-seguranca.example,NS,3600,"ns1.dnsbarato.example.",1.1.1.1\n2026-10-06T09:00:04Z,serraazul-seguranca.example,NS,3600,"ns2.dnsbarato.example.",1.1.1.1\n2026-10-06T09:00:04Z,coopserra-azul.example,NS,3600,"ns1.dnsbarato.example.",1.1.1.1\n2026-10-06T09:00:04Z,coopserra-azul.example,NS,3600,"ns2.dnsbarato.example.",1.1.1.1\n2026-10-06T09:00:04Z,coopserra-azul.example,A,300,"203.0.113.78",1.1.1.1\n2026-10-06T09:00:04Z,serraazul-seguranca.example,MX,300,"10 mail.serraazul-seguranca.example.",1.1.1.1\n', 'dns.stable.txt': 'coopserra-azul.example\tA\t203.0.113.78\ncoopserra-azul.example\tNS\tns1.dnsbarato.example.\ncoopserra-azul.example\tNS\tns2.dnsbarato.example.\nserraazul-pix.example\tA\t203.0.113.77\nserraazul-pix.example\tNS\tns1.dnsbarato.example.\nserraazul-pix.example\tNS\tns2.dnsbarato.example.\nserraazul-seguranca.example\tA\t203.0.113.78\nserraazul-seguranca.example\tMX\t10 mail.serraazul-seguranca.example.\nserraazul-seguranca.example\tNS\tns1.dnsbarato.example.\nserraazul-seguranca.example\tNS\tns2.dnsbarato.example.\nserraazul.example\tA\t198.51.100.20\nserraazul.example\tMX\t10 mx1.provedor-email.example.\nserraazul.example\tNS\tns1.provedor-dns.example.\nserraazul.example\tNS\tns2.provedor-dns.example.\nserraazul.example\tTXT\t"v=spf1 include:_spf.provedor-email.example include:envios.boletim.example ~all"\n', 'SHA256SUMS': '0b614cefa3dc28269d3b554aa9e08d2cf57dcfa75f00fd8b8590bda1d41cbabe  dns.csv\n4c94563f8a38bd0408fdcdbc9852c5ccc1cf7053101939dcb95af4021d88e254  dns.stable.txt\n'}}
RAW = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-13/snapshots"

origem = "GitHub"
for rodada, arquivos in EMBUTIDO.items():
    p = Path("snapshots-isca") / rodada
    p.mkdir(parents=True, exist_ok=True)
    for nome, conteudo in arquivos.items():
        try:
            with urllib.request.urlopen(f"{RAW}/{rodada}/{nome}", timeout=8) as r:
                conteudo = r.read().decode("utf-8")
        except Exception:
            origem = "cópia embutida"                       # plano B
        (p / nome).write_text(conteudo, encoding="utf-8")
print("Retratos gravados em snapshots-isca/ — origem:", origem)
print(sorted(x.name for x in Path("snapshots-isca").iterdir()))

In [ ]:
# ------------------------------------------------------------
# sha256sum -c SHA256SUMS, em Python
# ------------------------------------------------------------
def conferir(pasta):
    ok = True
    for linha in (pasta / "SHA256SUMS").read_text().splitlines():
        esperado, nome = linha.split(maxsplit=1)
        atual = hashlib.sha256((pasta / nome).read_bytes()).hexdigest()
        status = "OK" if atual == esperado else "FAILED"
        ok = ok and status == "OK"
        print(f"  {pasta.name}/{nome}: {status}")
    return ok

rodadas = sorted(Path("snapshots-isca").iterdir())
integros = all(conferir(p) for p in rodadas)
print("\nOs dois retratos estão íntegros?" , integros)

In [ ]:
# ------------------------------------------------------------
# diff "$a/dns.stable.txt" "$b/dns.stable.txt"
# Comparando conjuntos: o que só está em A (<) e o que só está em B (>)
# ------------------------------------------------------------
a, b = rodadas[0], rodadas[-1]
linhas_a = set((a / "dns.stable.txt").read_text().splitlines())
linhas_b = set((b / "dns.stable.txt").read_text().splitlines())

so_a = sorted(linhas_a - linhas_b)        # removidos
so_b = sorted(linhas_b - linhas_a)        # adicionados

# "Modificado" = mesmo domínio e tipo nos dois lados, com valor diferente
chaves_a = {tuple(l.split("\t")[:2]) for l in so_a}
mudancas = []
for l in so_a:
    d, t, v = l.split("\t")
    mudancas.append((d, t, v, "", "removido"))
for l in so_b:
    d, t, v = l.split("\t")
    tipo = "modificado" if (d, t) in chaves_a else "adicionado"
    mudancas.append((d, t, "", v, tipo))

print(f"Retrato A: {a.name}   Retrato B: {b.name}\n")
for l in so_a:
    print("<", l)
for l in so_b:
    print(">", l)
if not mudancas:
    print("no changes between", a.name, "and", b.name)

> 🧠 **Camada 8:** o diff diz *o que* mudou **entre 21:00:05Z de 5/out e 09:00:04Z de 6/out**. Não diz o instante, nem o porquê. "O domínio foi ativado às 3h da manhã" seria inventar.

## 3. 🎮 Quiz: o que o script faz aqui?

Seis perguntas sobre o comportamento do `dns-snapshot.sh`. Responda com a letra.

In [ ]:
MODO_JOGO = True

QUIZ = [
    ("O resolvedor cai no meio da rodada. Com set -euo pipefail, o script...",
     ["a) segue e grava o que conseguiu", "b) para na hora, sem gravar um retrato meio vazio", "c) tenta outro resolvedor"], "b",
     "-e para no primeiro erro e pipefail faz o dig | awk falhar se o dig falhar."),
    ("Você digitou $domian em vez de $domain. Com -u...",
     ["a) vira string vazia", "b) é erro e o script para", "c) o bash corrige"], "b",
     "-u trata variável não definida como erro."),
    ("Por que o diff é feito no dns.stable.txt e não no dns.csv?",
     ["a) porque o csv é maior", "b) porque TTL e hora da coleta mudam a cada rodada", "c) porque o diff não lê CSV"], "b",
     "Sem os campos voláteis, só sobram mudanças reais."),
    ("Para que serve o '|| [[ -n \"$line\" ]]' no while read?",
     ["a) pular linhas vazias", "b) não perder a última linha sem quebra de linha", "c) ler mais rápido"], "b",
     "Arquivos editados à mão muitas vezes terminam sem \\n."),
    ("O awk usa $4 == rt. O que isso evita?",
     ["a) gravar um CNAME como se fosse registro A", "b) gravar TTL errado", "c) consultar duas vezes"], "a",
     "A resposta a um A pode começar com o CNAME; o filtro fica só com o tipo pedido."),
    ("O cliente quer 'aproveitar o script' para varrer as portas de 203.0.113.0/24 toda noite. Você...",
     ["a) acrescenta o nmap no laço", "b) recusa: é ação ativa contra terceiros, fora do escopo", "c) faz só uma vez"], "b",
     "Automatizar não amplia a autorização. DNS público é passivo; varredura não é."),
]
acertos = 0
for i, (p, ops, certa, porque) in enumerate(QUIZ, start=1):
    print(f"\n[{i}/{len(QUIZ)}] {p}")
    for o in ops:
        print("   ", o)
    if MODO_JOGO:
        r = ""
        while r not in ("a", "b", "c"):
            r = input("   Resposta (a/b/c): ").strip().lower()
        acertos += (r == certa)
        print("   ✅" if r == certa else f"   ❌ Era {certa}.")
    print("   Por quê:", porque)
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}", "— Checkpoint BR cumprido." if acertos >= 5 else "— Meta: 5.")

## 4. 📝 O relatório de mudanças do caso

In [ ]:
#@title Relatório de mudanças — domínios da Operação Isca { display-mode: "form" }
autorizacao = "serraazul.example: site do cliente (contrato). Domínios imitação: infraestrutura do golpe nomeada no escopo (Dia 1)."  #@param {type:"string"}
causa_coopserra = "Domínio estacionado foi ativado em 203.0.113.78, vizinho do IP do site falso. Provável preparação de nova página (hipótese)."  #@param {type:"string"}
causa_mx = "Passou a aceitar e-mail: disparo por e-mail, receber respostas de vítimas, ou padrão do provedor (três hipóteses, nenhuma confirmada)."  #@param {type:"string"}
recomendacao = "Hoje: avisar atendimento e filtro de e-mail da Serra Azul sobre coopserra-azul.example e serraazul-seguranca.example."  #@param {type:"string"}

def hash_csv(p):
    return hashlib.sha256((p / "dns.csv").read_bytes()).hexdigest()

causas = {"coopserra-azul.example": causa_coopserra, "serraazul-seguranca.example": causa_mx}
tabela = "| Domínio | Tipo | Valor em A | Valor em B | Mudança | Causa provável |\n|---|---|---|---|---|---|\n"
for d, t, va, vb, tipo in mudancas:
    tabela += f"| {d} | {t} | {va or '—'} | {vb or '—'} | {tipo} | {causas.get(d, '')} |\n"

rel = f"""# Relatório de mudanças de DNS — Operação Isca (Dia 13)
Domínios monitorados: {autorizacao}
Retrato A: {a.name}, SHA-256 do dns.csv {hash_csv(a)}
Retrato B: {b.name}, SHA-256 do dns.csv {hash_csv(b)}
Conferência SHA256SUMS: {"OK nos dois" if integros else "FALHOU — não use estes retratos"}
Resolvedor: 1.1.1.1
Intervalo da mudança: entre {a.name} e {b.name} (UTC)

{tabela}
## Recomendação
{recomendacao}
"""
Path("notas").mkdir(exist_ok=True)
Path("notas/isca-dia13-mudancas.md").write_text(rel, encoding="utf-8")
print(rel)
try:
    from google.colab import files
    files.download("notas/isca-dia13-mudancas.md")
except ImportError:
    print("(Fora do Colab: o arquivo ficou em notas/)")

### (Opcional) O script Bash de verdade, no Colab

O Colab roda Ubuntu. Estas linhas instalam o `dig` e o ShellCheck e rodam o script do README contra `example.com`. Linhas com `!` são puladas pelo teste automático.

In [ ]:
!apt-get -qq install -y dnsutils shellcheck > /dev/null
!curl -sL https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/curriculum/phase-1-foundations/day-13.md -o day13.md
!awk '/^#!\/usr\/bin\/env bash/,/^echo "wrote/' day13.md > dns-snapshot.sh && chmod +x dns-snapshot.sh
!printf 'example.com\n' > domains.txt && ./dns-snapshot.sh domains.txt && shellcheck dns-snapshot.sh
print("(Se você está vendo só esta linha, os comandos de shell foram pulados.)")

## ✅ Antes de fechar o Dia 13

- [ ] Quiz com pelo menos 5/6
- [ ] `notas/isca-dia13-mudancas.md` com os dois hashes e as duas mudanças
- [ ] `dns-snapshot.sh`, duas pastas de retrato e `day13-change-report.md` na sua máquina (README, Passos 1 a 6)
- [ ] `status` do dia 13 = `estudado` no `curriculo/mapa-dos-dias.csv`

**Amanhã (Dia 14):** a tríade CID como aparece na evidência. A segunda entrega de logs da Serra Azul não bate com a primeira, e você vai descobrir qual propriedade isso fere.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>